# SVM Model Training and Evaluation

This notebook trains a Support Vector Machine (SVM) on the preprocessed CICEVSE2024 dataset. It includes data loading, comprehensive data visualisation, evaluation, and hyperparameter tuning using `GridSearchCV`.

**Dataset**: CICEVSE2024 Network Traffic (14 attack classes + benign)
**Algorithm**: LinearSVC (scikit-learn)
**Task**: Multiclass classification of EV charging network intrusions

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Plotting config
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='deep')

## 1. Load Data
Load the preprocessed train, validation, and test datasets. Note: ensure `preprocess.py` has been run previously.

In [ ]:
# Adjust path assuming the notebook runs from the project root or src/models/svm
import sys
if os.path.exists('../../../data/processed'):
    DATA_DIR = '../../../data/processed'
elif os.path.exists('data/processed'):
    DATA_DIR = 'data/processed'
else:
    DATA_DIR = '../data/processed' # fallback

print(f"Using data directory: {DATA_DIR}")

X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))
y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))

y_train_binary = y_train["Label_Binary"].values.ravel()
y_train_multi = y_train["Label_Multiclass"].values.ravel()
y_val_binary = y_val["Label_Binary"].values.ravel()
y_val_multi = y_val["Label_Multiclass"].values.ravel()
y_test_binary = y_test["Label_Binary"].values.ravel()
y_test_multi = y_test["Label_Multiclass"].values.ravel()

print("Data loaded successfully!")
print(f"X_train shape: {X_train.shape}")

## 1.1 Dataset Overview
Quick inspection of the training data: shape, data types, summary statistics, and missing value check.

In [ ]:
print(f"X_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"\nNumber of features: {X_train.shape[1]}")
print(f"\nData types:\n{X_train.dtypes.value_counts()}")
print(f"\nMissing values per column (if any):")
missing = X_train.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("  None — all features are clean.")
else:
    print(missing_cols)

print("\nSummary Statistics (first 10 features):")
X_train.iloc[:, :10].describe().round(3)

## 1.2 Train / Validation / Test Split Sizes
Visualise the data split proportions to verify the 70/15/15 split from preprocessing.

In [ ]:
split_sizes = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Samples': [len(X_train), len(X_val), len(X_test)]
})
split_sizes['Percentage'] = (split_sizes['Samples'] / split_sizes['Samples'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(split_sizes['Split'], split_sizes['Samples'], color=['#2196F3', '#FF9800', '#4CAF50'])
for bar, pct in zip(bars, split_sizes['Percentage']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,.0f} ({pct}%)', va='center', fontsize=11)
ax.set_xlabel('Number of Samples')
ax.set_title('Train / Validation / Test Split Sizes')
plt.tight_layout()
plt.show()

print(split_sizes.to_string(index=False))

## 1.3 Visualize Class Distributions
Before training the model, let's visualize the class distributions of our training dataset for both Binary and Multiclass targets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Binary Class Distribution
binary_counts = y_train['Label_Binary'].value_counts()
axes[0].bar(binary_counts.index.astype(str), binary_counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Binary Class Distribution (Train)', fontsize=13)
axes[0].set_xlabel('Label (0=Benign, 1=Attack)')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(binary_counts.items()):
    axes[0].text(i, val + 5000, f'{val:,}', ha='center', fontsize=10)

# Plot Multiclass Distribution
multi_counts = y_train['Label_Multiclass'].value_counts()
colors = sns.color_palette('viridis', len(multi_counts))
axes[1].barh(multi_counts.index, multi_counts.values, color=colors)
axes[1].set_title('Multiclass Distribution (Train)', fontsize=13)
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Attack Type')
for i, val in enumerate(multi_counts.values):
    axes[1].text(val + 1000, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nMulticlass label counts:")
print(multi_counts.to_string())

## 1.4 Feature Correlation Heatmap
Visualise the pairwise Pearson correlations of the top 30 features (by variance) to identify multicollinearity.

In [ ]:
# Select top 30 features by variance for readability
top_features = X_train.var().nlargest(30).index.tolist()
corr_matrix = X_train[top_features].corr()

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, fmt='.1f',
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})
plt.title('Feature Correlation Heatmap (Top 30 by Variance)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 1.5 Feature Distribution Box Plots
Box plots of the top 10 highest-variance features to visualise their range and spread after StandardScaler normalisation.

In [ ]:
top10 = X_train.var().nlargest(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(top10):
    # Sample for speed (1% of data is enough for distribution shape)
    sample = X_train[col].sample(n=min(10000, len(X_train)), random_state=42)
    axes[i].boxplot(sample.values, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#42A5F5', alpha=0.7))
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='x', labelbottom=False)

plt.suptitle('Top 10 Features by Variance — Box Plots (Scaled Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Hyperparameter Tuning (Binary Classification)
We use `LinearSVC` as it scales better to large datasets. We'll tune the `C` parameter (regularization strength) using cross-validation on the training set.

In [ ]:
print("Starting Hyperparameter Tuning for Binary Model...")
param_grid = {'C': [0.01, 0.1, 1, 10]}

svm_grid = GridSearchCV(LinearSVC(random_state=42, dual=False, max_iter=2000), 
                        param_grid, 
                        cv=3, 
                        scoring='f1', 
                        n_jobs=-1, 
                        verbose=2)

svm_grid.fit(X_train, y_train_binary)

print(f"Best Parameters: {svm_grid.best_params_}")
print(f"Best CV F1-Score: {svm_grid.best_score_:.4f}")

best_binary_model = svm_grid.best_estimator_

## 3. Evaluation Setup

In [ ]:
def evaluate_model(model, X, y, title_prefix="", is_multiclass=False):
    preds = model.predict(X)
    avg_method = 'weighted' if is_multiclass else 'binary'
    
    print(f"--- {title_prefix} Classification Report ---")
    print(classification_report(y, preds, zero_division=0))
    
    cm = confusion_matrix(y, preds)
    fig_size = max(6, len(np.unique(y)) * 0.8)
    plt.figure(figsize=(fig_size, fig_size * 0.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{title_prefix} Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    return preds

## 4. Evaluate Binary Model

In [ ]:
_ = evaluate_model(best_binary_model, X_val, y_val_binary, title_prefix="Binary Validation")
_ = evaluate_model(best_binary_model, X_test, y_test_binary, title_prefix="Binary Test")

## 5. Train & Evaluate Multiclass Model
For the multiclass model, we will use the best `C` parameter found during the binary search, or you can re-run `GridSearchCV`.

In [ ]:
best_C = svm_grid.best_params_['C']
print(f"Training Multiclass model with C={best_C}...")

best_multi_model = LinearSVC(C=best_C, random_state=42, dual=False, max_iter=2000)
best_multi_model.fit(X_train, y_train_multi)

_ = evaluate_model(best_multi_model, X_val, y_val_multi, title_prefix="Multiclass Validation", is_multiclass=True)
_ = evaluate_model(best_multi_model, X_test, y_test_multi, title_prefix="Multiclass Test", is_multiclass=True)

## 6. Save Models
Export the best models to the `saved_models` directory.

In [ ]:
if os.path.exists('../../../saved_models'):
    SAVE_DIR = '../../../saved_models'
elif os.path.exists('saved_models'):
    SAVE_DIR = 'saved_models'
else:
    SAVE_DIR = '../saved_models'

os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(best_binary_model, os.path.join(SAVE_DIR, "svm_model_binary.pkl"))
joblib.dump(best_multi_model, os.path.join(SAVE_DIR, "svm_model_multiclass.pkl"))

print("Models saved successfully!")